In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# TVP-VAR -> TVP-VMA -> GFEVD -> Connectedness
# ------------------------------------------------------------
# Input files in ./
#   - ./tvpvar_beta.npy
#   - ./tvpvar_cov.npy
#   - ./tvpvar_selected_lag.txt
#   - ./tvpvar_effective_dates.csv
#   - ./tvpvar_var_names.csv
#
# Output files in ./result
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# input files are in ./
BETA_FILE = BASE_DIR / "tvpvar_beta.npy"
COV_FILE = BASE_DIR / "tvpvar_cov.npy"
LAG_FILE = BASE_DIR / "tvpvar_selected_lag.txt"
DATES_FILE = BASE_DIR / "tvpvar_effective_dates.csv"
VARNAMES_FILE = BASE_DIR / "tvpvar_var_names.csv"

# output files are in ./result
OUT_LAST = RESULT_DIR / "gfevd_last.csv"
OUT_MEAN = RESULT_DIR / "gfevd_mean.csv"
OUT_ALL = RESULT_DIR / "gfevd_all.npy"
OUT_DIAG = RESULT_DIR / "gfevd_diag_summary.csv"

OUT_TCI = RESULT_DIR / "gfevd_tci_timeseries.csv"
OUT_TO = RESULT_DIR / "gfevd_directional_to.csv"
OUT_FROM = RESULT_DIR / "gfevd_directional_from.csv"
OUT_NET = RESULT_DIR / "gfevd_net.csv"
OUT_PAIRWISE_NET = RESULT_DIR / "gfevd_pairwise_net.csv"

H = 10

DEFAULT_VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
]

EPS = 1e-12
RIDGE_EPS = 1e-8
STABILITY_TOL = 0.999
VERBOSE_ERROR_LOG = True


# -----------------------------
# Helpers
# -----------------------------
def make_psd(mat, eps=RIDGE_EPS):
    mat = np.asarray(mat, dtype=float)

    if mat.ndim != 2 or mat.shape[0] != mat.shape[1]:
        raise ValueError(f"Sigma_t must be square 2D matrix, got shape={mat.shape}")

    if not np.all(np.isfinite(mat)):
        raise ValueError("Sigma_t contains non-finite values.")

    mat = 0.5 * (mat + mat.T)
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, eps, None)

    mat_psd = vecs @ np.diag(vals) @ vecs.T
    mat_psd = 0.5 * (mat_psd + mat_psd.T)

    return mat_psd


def load_var_names(k):
    if VARNAMES_FILE.exists():
        df_names = pd.read_csv(VARNAMES_FILE)

        if "var_name" not in df_names.columns:
            raise ValueError("tvpvar_var_names.csv must contain 'var_name' column.")

        var_names = df_names["var_name"].astype(str).tolist()

        if len(var_names) != k:
            raise ValueError(
                f"tvpvar_var_names.csv length({len(var_names)}) != k({k})"
            )

        return var_names

    if len(DEFAULT_VAR_NAMES) != k:
        raise ValueError(
            f"DEFAULT_VAR_NAMES length({len(DEFAULT_VAR_NAMES)}) != k({k}). "
            "Check variable count."
        )

    return DEFAULT_VAR_NAMES


def load_inputs():
    required_files = [BETA_FILE, COV_FILE, LAG_FILE]

    for path in required_files:
        if not path.exists():
            raise FileNotFoundError(f"Required input file not found: {path}")

    beta_t = np.load(BETA_FILE)
    cov_t = np.load(COV_FILE)

    with open(LAG_FILE, "r", encoding="utf-8") as f:
        p = int(f.read().strip())

    if beta_t.ndim != 3:
        raise ValueError(f"beta_t shape 이상: {beta_t.shape}")

    if cov_t.ndim != 3:
        raise ValueError(f"cov_t shape 이상: {cov_t.shape}")

    T_eff, k, kp = beta_t.shape

    if p < 1:
        raise ValueError("lag는 1 이상이어야 합니다.")

    if kp != k * p:
        raise ValueError(f"beta_t 마지막 차원({kp}) != k*p({k * p})")

    if cov_t.shape[0] != T_eff:
        raise ValueError(
            f"cov_t 시점 길이({cov_t.shape[0]}) != beta_t 시점 길이({T_eff})"
        )

    if cov_t.shape[1] != k or cov_t.shape[2] != k:
        raise ValueError(f"cov_t shape {cov_t.shape} is incompatible with k={k}")

    if not np.all(np.isfinite(beta_t)):
        raise ValueError("beta_t contains non-finite values.")

    if not np.all(np.isfinite(cov_t)):
        raise ValueError("cov_t contains non-finite values.")

    var_names = load_var_names(k)

    return beta_t, cov_t, p, T_eff, k, var_names


def load_dates(T_eff):
    if DATES_FILE.exists():
        df_dates = pd.read_csv(DATES_FILE)

        if "Date" not in df_dates.columns:
            raise ValueError("tvpvar_effective_dates.csv must contain 'Date' column.")

        if len(df_dates) != T_eff:
            raise ValueError(
                f"Date length mismatch: len(dates)={len(df_dates)} != T_eff={T_eff}"
            )

        return df_dates["Date"].astype(str).reset_index(drop=True)

    return pd.Series([str(i) for i in range(T_eff)], name="Date")


def unpack_A_mats(beta_one_t, p):
    beta_one_t = np.asarray(beta_one_t, dtype=float)

    if beta_one_t.ndim != 2:
        raise ValueError(f"beta_one_t must be 2D, got {beta_one_t.shape}")

    k, kp = beta_one_t.shape

    if kp != k * p:
        raise ValueError(f"beta_one_t shape mismatch: {beta_one_t.shape}, p={p}")

    A_list = []

    for lag in range(p):
        start = lag * k
        end = (lag + 1) * k
        A_lag = beta_one_t[:, start:end]
        A_list.append(A_lag)

    return A_list


def build_companion(A_list):
    if len(A_list) == 0:
        raise ValueError("A_list is empty.")

    k = A_list[0].shape[0]
    p = len(A_list)

    for idx, A in enumerate(A_list):
        if A.shape != (k, k):
            raise ValueError(
                f"A_list[{idx}] shape mismatch: {A.shape}, expected {(k, k)}"
            )

    if p == 1:
        return A_list[0].copy()

    top = np.hstack(A_list)
    bottom_left = np.eye(k * (p - 1))
    bottom_right = np.zeros((k * (p - 1), k))
    bottom = np.hstack([bottom_left, bottom_right])

    return np.vstack([top, bottom])


def spectral_radius_from_companion(comp):
    if not np.all(np.isfinite(comp)):
        raise ValueError("Companion matrix contains non-finite values.")

    vals = np.linalg.eigvals(comp)
    return float(np.max(np.abs(vals)))


def spectral_radius(A_list):
    comp = build_companion(A_list)
    return spectral_radius_from_companion(comp)


def stabilize_A_list(A_list, tol=STABILITY_TOL):
    rho_raw = spectral_radius(A_list)

    if rho_raw < tol:
        return A_list, rho_raw, rho_raw, 1.0

    scale = tol / max(rho_raw, EPS)
    A_stab = [A * scale for A in A_list]
    rho_used = spectral_radius(A_stab)

    return A_stab, rho_raw, rho_used, scale


def compute_phi_list(A_list, H):
    if H < 1:
        raise ValueError("H must be >= 1")

    p = len(A_list)
    k = A_list[0].shape[0]

    phi = [np.eye(k)]

    for h in range(1, H):
        phi_h = np.zeros((k, k))

        for j in range(1, min(h, p) + 1):
            phi_h += A_list[j - 1] @ phi[h - j]

        phi.append(phi_h)

    return phi


def safe_row_normalize(mat, eps=EPS):
    mat = np.asarray(mat, dtype=float)
    out = mat.copy()

    for i in range(out.shape[0]):
        s = out[i].sum()

        if (not np.isfinite(s)) or (s <= eps):
            out[i] = np.nan
        else:
            out[i] = out[i] / s

    return out


def compute_gfevd_one_t(A_list, Sigma_t, H):
    Sigma_t = make_psd(Sigma_t)

    A_list_stab, rho_raw, rho_used, scale_used = stabilize_A_list(
        A_list,
        tol=STABILITY_TOL,
    )

    k = Sigma_t.shape[0]
    phi_list = compute_phi_list(A_list_stab, H)

    gfevd = np.zeros((k, k), dtype=float)

    for i in range(k):
        e_i = np.zeros((k, 1))
        e_i[i, 0] = 1.0

        denom = 0.0

        for h in range(H):
            phi_h = phi_list[h]
            term = e_i.T @ phi_h @ Sigma_t @ phi_h.T @ e_i
            denom += float(term)

        denom = max(denom, EPS)

        for j in range(k):
            e_j = np.zeros((k, 1))
            e_j[j, 0] = 1.0

            sigma_jj = max(float(Sigma_t[j, j]), EPS)

            numer = 0.0

            for h in range(H):
                phi_h = phi_list[h]
                term = e_i.T @ phi_h @ Sigma_t @ e_j
                numer += float(term) ** 2

            gfevd[i, j] = (numer / sigma_jj) / denom

    gfevd = np.where(np.isfinite(gfevd), gfevd, np.nan)
    gfevd = safe_row_normalize(gfevd, eps=EPS)

    return gfevd, rho_raw, rho_used, scale_used


def compute_connectedness_one_t(gfevd_t):
    if np.isnan(gfevd_t).any():
        return None

    k = gfevd_t.shape[0]

    off_diag_sum = gfevd_t.sum() - np.trace(gfevd_t)
    tci = float(off_diag_sum / k * 100.0)

    directional_from = gfevd_t.sum(axis=1) - np.diag(gfevd_t)
    directional_to = gfevd_t.sum(axis=0) - np.diag(gfevd_t)
    net = directional_to - directional_from

    pairwise_net = np.zeros((k, k))

    for i in range(k):
        for j in range(k):
            if i == j:
                pairwise_net[i, j] = 0.0
            else:
                pairwise_net[i, j] = gfevd_t[j, i] - gfevd_t[i, j]

    return tci, directional_to, directional_from, net, pairwise_net


# -----------------------------
# Main
# -----------------------------
def main():
    beta_t, cov_t, p, T_eff, k, var_names = load_inputs()
    dates = load_dates(T_eff)

    print("==========================================")
    print("TVP-VAR -> GFEVD -> Connectedness")
    print("==========================================")
    print("T_eff :", T_eff)
    print("k     :", k)
    print("lag p :", p)
    print("H     :", H)
    print("dates :", "loaded" if DATES_FILE.exists() else "not found -> using t index")
    print("vars  :", var_names)
    print()

    gfevd_all = np.full((T_eff, k, k), np.nan)

    diag_rows = []
    tci_rows = []
    to_rows = []
    from_rows = []
    net_rows = []
    pairwise_rows = []

    fail_count = 0
    success_mask = np.zeros(T_eff, dtype=bool)

    for t in range(T_eff):
        error_msg = ""

        try:
            A_list_raw = unpack_A_mats(beta_t[t], p)

            gfevd_t, rho_raw, rho_used, scale_used = compute_gfevd_one_t(
                A_list_raw,
                cov_t[t],
                H,
            )

            if np.isnan(gfevd_t).any():
                raise ValueError("NaN detected in gfevd_t after normalization")

            row_err = float(np.max(np.abs(gfevd_t.sum(axis=1) - 1.0)))
            diag_mean = float(np.mean(np.diag(gfevd_t)))
            offdiag_mean = float(
                (gfevd_t.sum() - np.trace(gfevd_t)) / (k * (k - 1))
            )

            conn = compute_connectedness_one_t(gfevd_t)

            if conn is None:
                raise ValueError("Connectedness computation failed due to NaN GFEVD.")

            tci, directional_to, directional_from, net, pairwise_net = conn

            gfevd_all[t] = gfevd_t
            success_mask[t] = True

            tci_rows.append({
                "t": t,
                "Date": dates.iloc[t],
                "TCI": tci,
            })

            to_row = {"t": t, "Date": dates.iloc[t]}
            from_row = {"t": t, "Date": dates.iloc[t]}
            net_row = {"t": t, "Date": dates.iloc[t]}

            for i, name in enumerate(var_names):
                to_row[name] = directional_to[i]
                from_row[name] = directional_from[i]
                net_row[name] = net[i]

            to_rows.append(to_row)
            from_rows.append(from_row)
            net_rows.append(net_row)

            for i in range(k):
                for j in range(k):
                    if i != j:
                        pairwise_rows.append({
                            "t": t,
                            "Date": dates.iloc[t],
                            "from_var": var_names[i],
                            "to_var": var_names[j],
                            "net_pairwise": pairwise_net[i, j],
                        })

        except Exception as e:
            fail_count += 1
            error_msg = repr(e)

            try:
                A_list_raw = unpack_A_mats(beta_t[t], p)
                rho_raw = spectral_radius(A_list_raw)
            except Exception:
                rho_raw = np.nan

            rho_used = np.nan
            scale_used = np.nan
            row_err = np.nan
            diag_mean = np.nan
            offdiag_mean = np.nan
            tci = np.nan

            if VERBOSE_ERROR_LOG:
                print(f"[WARN] t={t} failed: {error_msg}")

        diag_rows.append({
            "t": t,
            "Date": dates.iloc[t],
            "rho_raw": rho_raw,
            "rho_used": rho_used,
            "scale_used": scale_used,
            "row_err": row_err,
            "diag_mean": diag_mean,
            "offdiag_mean": offdiag_mean,
            "tci": tci,
            "success": int(success_mask[t]),
            "error_msg": error_msg,
        })

        if (t + 1) % 200 == 0 or (t + 1) == T_eff:
            print(f"processed: {t + 1}/{T_eff} | failures so far: {fail_count}")

    if success_mask.sum() == 0:
        raise RuntimeError("All GFEVD computations failed. No valid output to summarize.")

    gfevd_last_idx = np.where(success_mask)[0][-1]
    gfevd_last = gfevd_all[gfevd_last_idx]
    gfevd_mean = np.nanmean(gfevd_all, axis=0)

    df_last = pd.DataFrame(gfevd_last, index=var_names, columns=var_names)
    df_mean = pd.DataFrame(gfevd_mean, index=var_names, columns=var_names)
    df_diag = pd.DataFrame(diag_rows)

    df_tci = pd.DataFrame(tci_rows)
    df_to = pd.DataFrame(to_rows)
    df_from = pd.DataFrame(from_rows)
    df_net = pd.DataFrame(net_rows)
    df_pairwise = pd.DataFrame(pairwise_rows)

    df_last.to_csv(OUT_LAST, encoding="utf-8-sig")
    df_mean.to_csv(OUT_MEAN, encoding="utf-8-sig")
    df_diag.to_csv(OUT_DIAG, index=False, encoding="utf-8-sig")
    np.save(OUT_ALL, gfevd_all)

    df_tci.to_csv(OUT_TCI, index=False, encoding="utf-8-sig")
    df_to.to_csv(OUT_TO, index=False, encoding="utf-8-sig")
    df_from.to_csv(OUT_FROM, index=False, encoding="utf-8-sig")
    df_net.to_csv(OUT_NET, index=False, encoding="utf-8-sig")
    df_pairwise.to_csv(OUT_PAIRWISE_NET, index=False, encoding="utf-8-sig")

    print()
    print("Saved files:")
    print(f" - {OUT_LAST}")
    print(f" - {OUT_MEAN}")
    print(f" - {OUT_ALL}")
    print(f" - {OUT_DIAG}")
    print(f" - {OUT_TCI}")
    print(f" - {OUT_TO}")
    print(f" - {OUT_FROM}")
    print(f" - {OUT_NET}")
    print(f" - {OUT_PAIRWISE_NET}")

    print()
    print(f"Last valid GFEVD index: t={gfevd_last_idx}, Date={dates.iloc[gfevd_last_idx]}")
    print(f"Successful periods: {int(success_mask.sum())}/{T_eff}")
    print(f"Total failures: {fail_count}")

    print()
    print("Last valid GFEVD:")
    print(df_last.round(4).to_string())

    print()
    print("Mean GFEVD:")
    print(df_mean.round(4).to_string())

    print()
    print("Diagnostic summary:")
    print(
        df_diag[
            [
                "rho_raw",
                "rho_used",
                "scale_used",
                "row_err",
                "diag_mean",
                "offdiag_mean",
                "tci",
                "success",
            ]
        ].describe().round(4).to_string()
    )


if __name__ == "__main__":
    main()

TVP-VAR -> GFEVD -> Connectedness
T_eff : 1031
k     : 9
lag p : 1
H     : 10
dates : not found -> using t index
vars  : ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'dlog_OIL', 'dlog_GAS', 'd_UST10Y', 'd_VIX']



C:\Users\User\AppData\Local\Temp\ipykernel_9964\3251201659.py:303: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  denom += float(term)
C:\Users\User\AppData\Local\Temp\ipykernel_9964\3251201659.py:318: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  numer += float(term) ** 2


processed: 200/1031 | failures so far: 0
processed: 400/1031 | failures so far: 0
processed: 600/1031 | failures so far: 0
processed: 800/1031 | failures so far: 0
processed: 1000/1031 | failures so far: 0
processed: 1031/1031 | failures so far: 0

Saved files:
 - result\gfevd_last.csv
 - result\gfevd_mean.csv
 - result\gfevd_all.npy
 - result\gfevd_diag_summary.csv
 - result\gfevd_tci_timeseries.csv
 - result\gfevd_directional_to.csv
 - result\gfevd_directional_from.csv
 - result\gfevd_net.csv
 - result\gfevd_pairwise_net.csv

Last valid GFEVD index: t=1030, Date=1030
Successful periods: 1031/1031
Total failures: 0

Last valid GFEVD:
             dlog_SOLVPN  dlog_COPPER  dlog_GOLD  dlog_SILVER  dlog_DXY  dlog_OIL  dlog_GAS  d_UST10Y   d_VIX
dlog_SOLVPN       0.4971       0.0674     0.0749       0.0286    0.0069    0.0985    0.0224    0.0073  0.1969
dlog_COPPER       0.0402       0.4304     0.1609       0.0996    0.0040    0.0307    0.1788    0.0483  0.0070
dlog_GOLD         0.0987   